In [ ]:
!pip install faiss-cpu sentence-transformers pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 36.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sde_MOSFET2D.txt to sde_MOSFET2D.txt
Saving sdevice_MOSFET2D.txt to sdevice_MOSFET2D.txt
Saving sdevice_ug.pdf to sdevice_ug.pdf
Saving sense_ug.pdf to sense_ug.pdf


In [ ]:
import fitz
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# =========================
# EMBEDDING MODEL
# =========================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# =========================
# LOADERS
# =========================
def load_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return " ".join([page.get_text() for page in doc])

def load_text_file(file_path):
    with open(file_path, "r") as f:
        return f.read()

def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

# =========================
# LOAD DATA
# =========================

# Manuals
sde_text = load_pdf("sense_ug.pdf")
sdevice_text = load_pdf("sdevice_ug.pdf")

# Sample scripts (UPLOAD THESE FILES)
sample_sde = load_text_file("sde_MOSFET2D.txt")
sample_sdevice = load_text_file("sdevice_MOSFET2D.txt")

# =========================
# BUILD CHUNKS
# =========================

manual_chunks, manual_tags = [], []
example_chunks, example_tags = [], []

# Manuals
for c in chunk_text(sde_text):
    manual_chunks.append(c)
    manual_tags.append("MANUAL_SDE")

for c in chunk_text(sdevice_text):
    manual_chunks.append(c)
    manual_tags.append("MANUAL_SDEVICE")

# Examples (VERY IMPORTANT)
for c in chunk_text(sample_sde):
    example_chunks.append(c)
    example_tags.append("EXAMPLE_SDE")

for c in chunk_text(sample_sdevice):
    example_chunks.append(c)
    example_tags.append("EXAMPLE_SDEVICE")

# Combine
all_chunks = manual_chunks + example_chunks
all_tags = manual_tags + example_tags

# =========================
# EMBEDDING + FAISS
# =========================
embeddings = embed_model.encode(all_chunks, batch_size=32, show_progress_bar=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("✅ RAG index ready with manuals + examples!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

✅ RAG index ready with manuals + examples!


In [ ]:
# =========================
# GEMINI SETUP
# =========================
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('Gemini_API_Key')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-flash-latest')

# =========================
# RETRIEVAL FUNCTION
# =========================
def retrieve(query, k_manual=5, k_example=3):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k_manual + k_example)

    manual_context = []
    example_context = []

    for i in I[0]:
        tag = all_tags[i]
        if "EXAMPLE" in tag and len(example_context) < k_example:
            example_context.append(f"[{tag}] {all_chunks[i]}")
        elif "MANUAL" in tag and len(manual_context) < k_manual:
            manual_context.append(f"[{tag}] {all_chunks[i]}")

    return "\n\n".join(manual_context), "\n\n".join(example_context)

# =========================
# RAG QUERY
# =========================
def query_rag(user_query):

    manual_ctx, example_ctx = retrieve(user_query)

    prompt = f"""
You are a Synopsys Sentaurus TCAD expert.

You have:
1. Documentation (for syntax)
2. Example scripts (for structure and correctness)

---------------------
EXAMPLES (STRICT TEMPLATE)
---------------------
{example_ctx}

---------------------
DOCUMENTATION
---------------------
{manual_ctx}

---------------------
TASK
---------------------
Generate complete and runnable scripts:

1. SDE script
2. SDevice script

---------------------
STRICT REQUIREMENTS
---------------------

SDE:
- MUST define: substrate, source, drain, channel, oxide, gate
- MUST include: LDD + Halo doping
- MUST include: mesh + contacts

SDevice:
- MUST include Physics models (Mobility, SRH, Fermi)
- MUST include Solve block
- MUST include Id-Vg sweep

RULES:
- Follow example structure EXACTLY
- Do NOT skip regions (source/drain MUST exist)
- Use realistic nanoscale MOSFET values
- No explanations

---------------------
USER REQUEST
---------------------
{user_query}

---------------------
OUTPUT FORMAT
---------------------

### SDE SCRIPT
<code>

### SDEVICE SCRIPT
<code>
"""

    response = gemini_model.generate_content(
        prompt,
        generation_config={"temperature": 0.2},
        request_options={'timeout': 1200}
    )

    return response.text if response.text else "⚠️ No response"

# =========================
# VALIDATOR (UNCHANGED)
# =========================
def validate(script):
    errors = []
    if "Physics {" not in script:
        errors.append("Missing Physics block")
    if "Solve {" not in script:
        errors.append("Missing Solve block")
    if "contact" not in script.lower():
        errors.append("Missing contacts")
    return errors



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
query = "Generate a 35nm NMOSFET with Halo doping and IdVg sweep"

output = query_rag(query)
print(output)

### SDE SCRIPT
<code>
(sde:clear)

;; =========================
;; PARAMETERS (um units)
;; =========================
(define Lg 0.035)        ; 35 nm gate
(define Lsd 0.030)      ; source/drain length
(define Ltot (+ Lg (* 2 Lsd))) ; total length
(define Tsub 0.2)       ; substrate depth
(define Tch 0.015)      ; channel thickness
(define Tox 0.0015)     ; 1.5 nm oxide
(define Tgate 0.03)     ; metal gate height

;; =========================
;; GEOMETRY
;; =========================
;; Substrate
(sdegeo:create-rectangle (position 0 (- Tsub) 0) (position Ltot 0 0) "Silicon" "Substrate")

;; Source
(sdegeo:create-rectangle (position 0 0 0) (position Lsd Tch 0) "Silicon" "Source")

;; Drain
(sdegeo:create-rectangle (position (- Ltot Lsd) 0 0) (position Ltot Tch 0) "Silicon" "Drain")

;; Channel
(sdegeo:create-rectangle (position Lsd 0 0) (position (- Ltot Lsd) Tch 0) "Silicon" "Channel")

;; Gate oxide
(sdegeo:create-rectangle (position Lsd Tch 0) (position (- Ltot Lsd) (+ Tch Tox) 0) "Si

In [ ]:
import pickle

faiss.write_index(index, "faiss_index.bin")

with open("chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

with open("tags.pkl", "wb") as f:
    pickle.dump(all_tags, f)

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 60.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import fitz
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

# =========================
# CONFIG
# =========================
st.set_page_config(page_title="TCAD RAG Generator", layout="wide")

# =========================
# LOAD MODELS
# =========================
@st.cache_resource
def load_models():
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    return embed_model

embed_model = load_models()

# =========================
# LOAD DATA (FAISS + chunks)
# =========================
@st.cache_resource
def load_index():
    index = faiss.read_index("faiss_index.bin")

    with open("chunks.pkl", "rb") as f:
        all_chunks = pickle.load(f)

    with open("tags.pkl", "rb") as f:
        all_tags = pickle.load(f)

    return index, all_chunks, all_tags

index, all_chunks, all_tags = load_index()

# =========================
# GEMINI SETUP
# =========================
import os
# Sidebar input for API key
st.sidebar.title("🔑 API Configuration")

api_key = st.sidebar.text_input(
    "Enter your Gemini API Key",
    type="password"
)

# Stop app if no key
if not api_key:
    st.warning("Please enter your API key to continue")
    st.stop()

# Configure Gemini
genai.configure(api_key=api_key)
gemini_model = genai.GenerativeModel('gemini-flash-latest')

# =========================
# RETRIEVAL
# =========================
def retrieve(query, k_manual=5, k_example=3):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k_manual + k_example)

    manual_context = []
    example_context = []

    for i in I[0]:
        tag = all_tags[i]
        if "EXAMPLE" in tag and len(example_context) < k_example:
            example_context.append(f"[{tag}] {all_chunks[i]}")
        elif "MANUAL" in tag and len(manual_context) < k_manual:
            manual_context.append(f"[{tag}] {all_chunks[i]}")

    return "\n\n".join(manual_context), "\n\n".join(example_context)

# =========================
# GENERATION
# =========================
def query_rag(user_query):

    manual_ctx, example_ctx = retrieve(user_query)

    prompt = f"""
You are a Synopsys Sentaurus TCAD expert.

EXAMPLES:
{example_ctx}

DOCUMENTATION:
{manual_ctx}

TASK:
Generate complete scripts.

SDE:
- substrate, source, drain, channel, oxide, gate
- LDD + Halo
- mesh + contacts

SDevice:
- Physics (Mobility, SRH, Fermi)
- Solve
- Id-Vg sweep

RULES:
- Follow example structure
- No explanation

USER:
{user_query}

OUTPUT:

### SDE SCRIPT
### SDEVICE SCRIPT
"""

    response = gemini_model.generate_content(
        prompt,
        generation_config={"temperature": 0.2}
    )

    return response.text if response.text else "⚠️ No response"

# =========================
# VALIDATOR
# =========================
def validate(script):
    errors = []
    if "Physics {" not in script:
        errors.append("Missing Physics block")
    if "Solve {" not in script:
        errors.append("Missing Solve block")
    if "contact" not in script.lower():
        errors.append("Missing contacts")
    return errors

# =========================
# UI
# =========================
st.title("🧠 TCAD Script Generator (RAG + Gemini)")

col1, col2 = st.columns(2)

with col1:
    device = st.selectbox("Device Type", ["NMOS", "PMOS"])
    node = st.selectbox("Technology Node", ["45nm", "35nm", "28nm"])
    halo = st.checkbox("Include Halo Doping", True)

with col2:
    user_query = st.text_area(
        "Describe your device",
        f"Generate a {node} {device} with LDD and halo doping and Id-Vg sweep"
    )

if st.button("🚀 Generate Scripts"):

    with st.spinner("Generating TCAD scripts..."):
        output = query_rag(user_query)

    st.success("Done!")

    # Split output
    if "### SDEVICE SCRIPT" in output:
        sde, sdevice = output.split("### SDEVICE SCRIPT")
    else:
        sde, sdevice = output, ""

    # Display scripts
    st.subheader("📄 SDE Script")
    st.code(sde, language="python")

    st.subheader("📄 SDevice Script")
    st.code(sdevice, language="python")

    # Download buttons
    st.download_button("Download SDE", sde, "sde.cmd")
    st.download_button("Download SDevice", sdevice, "sdevice.cmd")

    # Validation
    st.subheader("🧪 Validation")
    errors = validate(output)

    if errors:
        for e in errors:
            st.error(e)
    else:
        st.success("No major issues detected ✅")

Writing app.py


In [ ]:
!ls

app.py	    faiss_index.bin  sde_MOSFET2D.txt	   sdevice_ug.pdf  tags.pkl
chunks.pkl  sample_data      sdevice_MOSFET2D.txt  sense_ug.pdf


In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "Gemini_API_Key"

In [ ]:
# Install pyngrok
!pip install pyngrok

You will need an `ngrok` authentication token to proceed. You can get one from [ngrok's website](https://ngrok.com/signup). Once you have it, add it to Colab secrets under the name `NGROK_AUTH_TOKEN` (similar to how you added `Gemini_API_Key`).

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

# Get ngrok auth token from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok authenticated!")

ngrok authenticated!


Now, we will save your Streamlit application code into a Python file and run it in the background. Then, `ngrok` will create a public URL that you can use to access your Streamlit app.

In [ ]:
!streamlit run app.py &>/dev/null &
from pyngrok import ngrok
print(ngrok.connect(8501))

NgrokTunnel: "https://procedure-busybody-wilt.ngrok-free.dev" -> "http://localhost:8501"
